In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Load data
df_train = pd.read_csv('train.csv')
df_train.head()

In [ ]:
df_train.info()

In [ ]:
df_train.describe(include="all")

In [ ]:
# Split data
X = df_train.drop('Y', axis=1)
y = df_train['Y']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Cek ukuran data:")
print(X_train.shape, X_val.shape)

In [ ]:
# Definisi kolom kategorikal dan numerik
cat_cols = ['X2', 'X3', 'X4']
num_cols = [col for col in X.columns if col not in cat_cols]

In [ ]:
# Penanganan outlier
outlier_bounds = {}

for col in num_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outlier_bounds[col] = (lower, upper)

    # Capit Train & Val
    X_train[col] = X_train[col].clip(lower=lower, upper=upper)
    X_val[col] = X_val[col].clip(lower=lower, upper=upper)

In [ ]:
# Preprocessing (StandardScaler & OneHot)
# 1. Tipe numerik
imputer_num = SimpleImputer(strategy='median')
scaler = StandardScaler()

imputer_num.fit(X_train[num_cols])
train_num = scaler.fit_transform(imputer_num.transform(X_train[num_cols]))
val_num = scaler.transform(imputer_num.transform(X_val[num_cols]))

# 2. Tipe kategorikal
imputer_cat = SimpleImputer(strategy='most_frequent')
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

imputer_cat.fit(X_train[cat_cols])
train_cat = encoder.fit_transform(imputer_cat.transform(X_train[cat_cols]))
val_cat = encoder.transform(imputer_cat.transform(X_val[cat_cols]))

# 3. Menggabungkan keduanya
X_train_final = np.hstack([train_num, train_cat])
X_val_final = np.hstack([val_num, val_cat])

In [ ]:
# Modeling

# Hitung rasio otomatis untuk menangani imbalance
ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    scale_pos_weight=ratio,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

model.fit(X_train_final, y_train)

In [ ]:
# Evaluasi
print("Evaluasi:")
y_pred = model.predict(X_val_final)
y_prob = model.predict_proba(X_val_final)[:, 1]
print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))
print("\nClassification Report:")
print(classification_report(y_val, y_pred))
print(f"ROC-AUC Score: {roc_auc_score(y_val, y_prob):.4f}")

In [ ]:
# Prediksi file test

df_test = pd.read_csv('test.csv')

for col in num_cols:
    lower, upper = outlier_bounds[col]
    df_test[col] = df_test[col].clip(lower=lower, upper=upper)

test_num = scaler.transform(imputer_num.transform(df_test[num_cols]))
test_cat = encoder.transform(imputer_cat.transform(df_test[cat_cols]))
X_test_final = np.hstack([test_num, test_cat])

final_pred = model.predict(X_test_final)

sub = pd.DataFrame()
if 'ID' in df_test.columns: sub['ID'] = df_test['ID']
else: sub['ID'] = df_test.index

sub['Y'] = final_pred
sub.to_csv('Ridho_Abdee_Nugraha-programming_1-ADIIP2026A.csv', index=False)
print("File 'Ridho_Abdee_Nugraha-programming_1-ADIIP2026A.csv' sudah siap!")